# Dotlas US restaurants (Databricks Marketplace)

**Setup:** `pip install -r requirements.txt` from the **Nuri_App** folder.

**Credentials:** Put `DATABRICKS_HOST`, `DATABRICKS_HTTP_PATH`, and `DATABRICKS_TOKEN` in `.env` at the repo root (the next cell loads them). Or export them in the shell / Jupyter kernel env.

**Token (fixes `required scopes: sql`):** In **Settings → Developer → Access tokens → Generate new token**, either leave **API scopes empty** (token gets full workspace permissions, including SQL), **or** add the **`sql`** scope explicitly. A *scoped* token without `sql` will fail for this notebook. After changing `.env`, re-run the **first** code cell (`load_dotenv` uses `override=True` so the new token loads).

**Kernel working directory:** Open/run Jupyter with working directory **Nuri_App** (folder with `package.json`) so `.env` and `src/lib/databricks_client.py` are found.

In [10]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "src" / "lib" / "databricks_client.py").exists():
        return cwd
    if (cwd.parent / "src" / "lib" / "databricks_client.py").exists():
        return cwd.parent
    raise FileNotFoundError(
        "Could not find src/lib/databricks_client.py. "
        "Open/run Jupyter with working directory set to Nuri_App (repo root)."
    )


ROOT = _repo_root()
# override=True: updating .env and re-running this cell replaces env vars (default dotenv does not).
load_dotenv(ROOT / ".env", override=True)
_lib = ROOT / "src" / "lib"
if str(_lib) not in sys.path:
    sys.path.insert(0, str(_lib))

import importlib

import databricks_client
importlib.reload(databricks_client)
from databricks_client import DatabricksClient

# When CITY is set: ROWS=None loads every matching row (no SQL LIMIT).
# When CITY is None: set ROWS to a number (random US sample).
ROWS = None
CITY = "San Francisco"  # set to None for a random US sample (no city filter)

In [11]:
host = os.environ.get("DATABRICKS_HOST")
http_path = os.environ.get("DATABRICKS_HTTP_PATH")
token = os.environ.get("DATABRICKS_TOKEN")
if not host or not http_path or not token:
    raise RuntimeError(
        "Set DATABRICKS_HOST, DATABRICKS_HTTP_PATH, and DATABRICKS_TOKEN in your environment."
    )

client = DatabricksClient(host, http_path, token)
df = client.get_dotlas_restaurants(limit=ROWS, city=CITY)
df.shape

ThriftBackend.attempt_request: Exception: 


RequestError: Error during request to server: : Provided PAT token does not have required scopes: sql. 

In [3]:
df.head(10)

NameError: name 'df' is not defined

In [ ]:
df.info()

## Exploration

Add cells below for filters, `value_counts`, plots, exports, etc.

In [ ]:
# Example: with CITY="San Francisco" above, `df` is already SF-only.
# If you used CITY=None, filter with: df[df["city"].str.lower() == "san francisco"]
df.head(50)